Проведем анализ зарубежного рынка авторского кино.
Для этого:
1) запарсим сайты:
- Cannes winners
- Berlinale
- Venice
2) Подключим TMDb API.
3) Составим dataset признанных зарубежных фильмов

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

In [3]:
#проверим, открывает ли пайтон сайт Канн
url = "https://www.festival-cannes.com/en/press/press-releases/the-78th-festival-de-cannes-winners-list/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

print(response.status_code)
print(response.text[:500])

200
<!doctype html>
<html lang="en-US" class="no-js">

<head>
	<meta charset="UTF-8">
	<meta name="viewport" content="width=device-width, initial-scale=1" />
	<link rel="preconnect" href="https://service.mtcaptcha.com">
	<link rel="preconnect" href="https://service2.mtcaptcha.com">	
		<link rel="preload" as="font" crossorigin="anonymous" type="font/woff2"
		href="https://www.festival-cannes.com/wp-content/themes/fdc/fonts/Everett-light.woff2" />
		<link rel="preload" as="font" crossorigin="anonymous


In [4]:
#превращаем страницу в текст
soup = BeautifulSoup(response.text, "html.parser")
text = soup.get_text(separator="\n", strip=True)

lines = [line.strip() for line in text.split("\n") if line.strip()]

for i, line in enumerate(lines[:120]):
    print(i, line)

0 The 78th Festival de Cannes winners' list - Festival de Cannes
1 Go to the main navigation
2 Go to the content
3 Go to search
4 Festival de Cannes
5 Festival de Cannes
6 OFFICIAL SELECTION
7 CINÉMA DE DEMAIN
8 MARCHÉ DU FILM
9 My Account
10 Online ticket office
11 FR
12 EN
13 Festival de Cannes
14 News
15 Live & Vidéos
16 Selection
17 Juries
18 Awards
19 Events
20 Immersive Competition
21 Your Festival Experience
22 Press area
23 Hide the search
24 Show the main navigation
25 Search
26 Press area
27 Journalists, please enter your password to unlock the content reserved for you.
28 OK
29 Press releases
30 Accreditation
31 Practical Guide
32 Press Material
33 Press conferences
34 FAQ
35 Locked content
36 This content is exclusively reserved for the press. Please enter your password.
37 OK
38 If you are not a journalist, please see
39 the programme
40 reserved for festival-goers.
41 Previous page
42 The 78th Festival de Cannes winners’ list
43 Event
44 published on 24.05.2025
45 Share
4

In [11]:
#делаем функцию для скачивания текста страницы
def get_page_lines(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)

    print(url, response.status_code)

    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator="\n", strip=True)
    lines = [line.strip() for line in text.split("\n") if line.strip()]

    return lines

In [14]:
#добавляем ссылки на сайты разных лет
urls = {
    2023: "https://www.festival-cannes.com/en/press/press-releases/the-76th-festival-de-cannes-winners-list/",
    2024: "https://www.festival-cannes.com/en/press/press-releases/the-77th-festival-de-cannes-winners-list/",
    2025: "https://www.festival-cannes.com/en/press/press-releases/the-78th-festival-de-cannes-winners-list/"
}

all_lines = {}

for year, url in urls.items():
    all_lines[year] = get_page_lines(url)

all_lines

https://www.festival-cannes.com/en/press/press-releases/the-76th-festival-de-cannes-winners-list/ 200
https://www.festival-cannes.com/en/press/press-releases/the-77th-festival-de-cannes-winners-list/ 200
https://www.festival-cannes.com/en/press/press-releases/the-78th-festival-de-cannes-winners-list/ 200


{2023: ["The 76th Festival de Cannes winners' list - Festival de Cannes",
  'Go to the main navigation',
  'Go to the content',
  'Go to search',
  'Festival de Cannes',
  'Festival de Cannes',
  'OFFICIAL SELECTION',
  'CINÉMA DE DEMAIN',
  'MARCHÉ DU FILM',
  'My Account',
  'Online ticket office',
  'FR',
  'EN',
  'Festival de Cannes',
  'News',
  'Live & Vidéos',
  'Selection',
  'Juries',
  'Awards',
  'Events',
  'Immersive Competition',
  'Your Festival Experience',
  'Press area',
  'Hide the search',
  'Show the main navigation',
  'Search',
  'Press area',
  'Journalists, please enter your password to unlock the content reserved for you.',
  'OK',
  'Press releases',
  'Accreditation',
  'Practical Guide',
  'Press Material',
  'Press conferences',
  'FAQ',
  'Locked content',
  'This content is exclusively reserved for the press. Please enter your password.',
  'OK',
  'If you are not a journalist, please see',
  'the programme',
  'reserved for festival-goers.',
  'Previou

In [23]:
# Список наград, которые мы хотим парсить
AWARD_MAP = {
    "PALME D’OR": "Palme d'Or",
    "PALME D'OR": "Palme d'Or",
    "Palme d’or": "Palme d'Or",
    "GRAND PRIX": "Grand Prix",
    "Grand Prix": "Grand Prix",
    "JURY PRIZE": "Jury Prize",
    "Jury Prize": "Jury Prize",
    "Joint Jury Prize": "Joint Jury Prize",
    "BEST DIRECTOR": "Best Director",
    "Best Director": "Best Director",
    "BEST SCREENPLAY": "Best Screenplay",
    "Best Screenplay": "Best Screenplay",
    "SPECIAL AWARD": "Special Award",
    "Special Award": "Special Award",
    "BEST PERFORMANCE BY AN ACTRESS": "Best Performance by an Actress",
    "Best performance by an actress": "Best Performance by an Actress",
    "BEST PERFORMANCE BY AN ACTOR": "Best Performance by an Actor",
    "Best performance by an actor": "Best Performance by an Actor",
}


In [24]:
#берем блок полнометражных фильмов
def extract_feature_block(lines):
    start = lines.index("Feature Films")

    end = None
    for marker in ["Short films", "Short Films"]:
        if marker in lines:
            end = lines.index(marker)
            break

    return lines[start:end]

In [25]:
def clean_block(block):
    trash = ["Feature Films", "2023 winners' list", "2024 winners' list", "2025 winners' list"]
    return [x for x in block if x not in trash]


def get_english_title(original_title):
    match = re.search(r"\((.*?)\)", original_title)
    if match:
        return match.group(1).strip()
    return original_title.strip()

In [26]:
def collect_award_segments(feature_lines):
    segments = []
    current_award = None
    current_lines = []

    for line in feature_lines:
        if line in AWARD_MAP:
            if current_award is not None:
                segments.append((current_award, current_lines))

            current_award = AWARD_MAP[line]
            current_lines = []
        else:
            if current_award is not None:
                current_lines.append(line)

    if current_award is not None:
        segments.append((current_award, current_lines))

    return segments


In [36]:
 # Читает название фильма. Если следующая строка — английское название в скобках, приклеивает его к оригиналу.
def read_title(lines, idx):

    title = lines[idx]

    if idx + 1 < len(lines) and lines[idx + 1].startswith("("):
        title = title + " " + lines[idx + 1]
        next_idx = idx + 2
    else:
        next_idx = idx + 1

    return title, get_english_title(title), next_idx

In [32]:
def parse_segment(year, award, segment):
    rows = []

    segment = [x for x in segment if x not in ["d", "irected by"]]

    # Награды, где сначала идет фильм, потом человек
    if award in ["Palme d'Or", "Grand Prix", "Jury Prize", "Joint Jury Prize"]:
        i = 0

        while i < len(segment):
            film_original, film_english, next_i = read_title(segment, i)

            winner = None
            director = None

            if next_i < len(segment):
                person_line = segment[next_i]
                if person_line.startswith("directed by "):
                    director = person_line.replace("directed by ", "")
                    winner = director
                    next_i += 1
                else:
                    winner = person_line
                    director = person_line
                    next_i += 1

            rows.append({
                "festival": "Cannes",
                "year": year,
                "award": award,
                "film_original": film_original,
                "film_english": film_english,
                "award_winner": winner,
                "director": director
            })

            i = next_i

    # Special Award бывает в двух форматах
    elif award == "Special Award":
        if "for" in segment:
            marker = segment.index("for")
            winner = " ".join(segment[:marker])
            film_original, film_english, _ = read_title(segment, marker + 1)
            director = winner
        else:
            film_original, film_english, next_i = read_title(segment, 0)
            winner = segment[next_i] if next_i < len(segment) else None
            director = winner

        rows.append({
            "festival": "Cannes",
            "year": year,
            "award": award,
            "film_original": film_original,
            "film_english": film_english,
            "award_winner": winner,
            "director": director
        })

    # Награды, где сначала человек, потом for/in, потом фильм
    elif award in [
        "Best Director",
        "Best Screenplay",
        "Best Performance by an Actress",
        "Best Performance by an Actor"
    ]:
        marker = None

        if "for" in segment:
            marker = "for"
        elif "in" in segment:
            marker = "in"

        if marker:
            marker_idx = segment.index(marker)
            winner = " ".join(segment[:marker_idx])
            film_original, film_english, next_i = read_title(segment, marker_idx + 1)

            director = None
            if next_i < len(segment) and segment[next_i].startswith("directed by "):
                director = segment[next_i].replace("directed by ", "")

            rows.append({
                "festival": "Cannes",
                "year": year,
                "award": award,
                "film_original": film_original,
                "film_english": film_english,
                "award_winner": winner,
                "director": director
            })

    return rows


def parse_cannes_year(lines, year):
    block = extract_feature_block(lines)
    block = clean_block(block)
    segments = collect_award_segments(block)

    year_rows = []

    for award, segment in segments:
        year_rows.extend(parse_segment(year, award, segment))

    return year_rows


all_rows = []

for year in [2023, 2024, 2025]:
    all_rows.extend(parse_cannes_year(all_lines[year], year))

cannes_winners_df = pd.DataFrame(all_rows)

cannes_winners_df = cannes_winners_df.drop_duplicates(
    subset=["year", "award", "film_english"]
).reset_index(drop=True)

cannes_winners_df

,festival,year,award,film_original,film_english,award_winner,director
0,Cannes,2023,Palme d'Or,ANATOMIE D’UNE CHUTE (ANATOMY OF A FALL),ANATOMY OF A FALL,Justine TRIET,Justine TRIET
1,Cannes,2023,Grand Prix,THE ZONE OF INTEREST,THE ZONE OF INTEREST,Jonathan GLAZER,Jonathan GLAZER
2,Cannes,2023,Best Director,LA PASSION DE DODIN BOUFFANT (THE POT-AU-FEU),THE POT-AU-FEU,TRAN ANH Hùng,None
3,Cannes,2023,Jury Prize,KUOLLEET LEHDET (FALLEN LEAVES),FALLEN LEAVES,irected by Aki KAURISMÄKI,irected by Aki KAURISMÄKI
4,Cannes,2023,Best Screenplay,KAIBUTSU (MONSTER),MONSTER,SAKAMOTO Yuji,KORE-EDA Hirokazu
5,Cannes,2023,Best Performance by an Actress,KURU OTLAR USTUNE (ABOUT DRY GRASSES),ABOUT DRY GRASSES,Merve DIZDAR,Nuri Bilge CEYLAN
6,Cannes,2023,Best Performance by an Actor,PERFECT DAYS,PERFECT DAYS,Koji YAKUSHO,Wim WENDERS
7,Cannes,2024,Palme d'Or,ANORA,ANORA,Sean BAKER,Sean BAKER
8,Cannes,2024,Grand Prix,ALL WE IMAGINE AS LIGHT,ALL WE IMAGINE AS LIGHT,Payal KAPADIA,Payal KAPADIA
9,Cannes,2024,Jury Prize,EMILIA PÉREZ,EMILIA PÉREZ,Jacques AUDIARD,Jacques AUDIARD
